# Find Metric Montage Candidates

This notebook reads `shot_length_results/shot_lengths_wide.csv`, checks consecutive shot lengths for a metric relation, and quotes `shot_lengths_long.csv` to recover exact shot numbers and time ranges.

Default definition: a candidate is a consecutive run of at least `MIN_CONSECUTIVE_SHOTS` shots where every shot length is within `TOLERANCE_SECONDS` of a positive integer multiple of `BASE_LENGTH_SECONDS`. For example, `MIN_CONSECUTIVE_SHOTS = 3`, `BASE_LENGTH_SECONDS = 3.0`, and `TOLERANCE_SECONDS = 0.2` finds runs where each shot is approximately `3, 6, 9, 12...` seconds.

In [1]:
from __future__ import annotations

import math
import os
import re
from pathlib import Path

import pandas as pd

SHOT_LENGTH_RESULTS_DIR = Path("./shot_length_results")
WIDE_CSV = SHOT_LENGTH_RESULTS_DIR / "shot_lengths_wide.csv"
LONG_CSV = SHOT_LENGTH_RESULTS_DIR / "shot_lengths_long.csv"

OUTPUT_DIR = Path("./metric_montage_results")

# Example output suffix: shot3_len3_tlr0p2
MIN_CONSECUTIVE_SHOTS = 3
BASE_LENGTH_SECONDS = 3.0
TOLERANCE_SECONDS = 0.2

# "maximal_runs" records each longest uninterrupted matching run once.
# "all_windows" records every matching sliding window of exactly MIN_CONSECUTIVE_SHOTS.
RECORD_MODE = "maximal_runs"

WRITE_PER_FILM_CSV = True
WRITE_COMBINED_CSV = True

OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

print(f"Wide CSV: {WIDE_CSV.resolve()}")
print(f"Long CSV: {LONG_CSV.resolve()}")
print(f"Output directory: {OUTPUT_DIR.resolve()}")

Wide CSV: /Users/caoshuangyuan/Desktop/Workspace/DirtyPilot/shot_length_results/shot_lengths_wide.csv
Long CSV: /Users/caoshuangyuan/Desktop/Workspace/DirtyPilot/shot_length_results/shot_lengths_long.csv
Output directory: /Users/caoshuangyuan/Desktop/Workspace/DirtyPilot/metric_montage_results


In [2]:
def decimal_token(value: float | int) -> str:
    text = f"{float(value):g}"
    return text.replace(".", "p").replace("-", "m")


def safe_filename_part(text: str) -> str:
    text = str(text).strip()
    text = re.sub(r"[\\/:*?\"<>|]", "_", text)
    return text or "untitled"


def output_suffix() -> str:
    return (
        f"shot{MIN_CONSECUTIVE_SHOTS}"
        f"_len{decimal_token(BASE_LENGTH_SECONDS)}"
        f"_tlr{decimal_token(TOLERANCE_SECONDS)}"
    )


def shot_column_number(column_name: str) -> int:
    match = re.fullmatch(r"shot_(\d+)", column_name)
    if not match:
        raise ValueError(f"Not a shot column: {column_name}")
    return int(match.group(1))


def sorted_shot_columns(wide_df: pd.DataFrame) -> list[str]:
    columns = [column for column in wide_df.columns if re.fullmatch(r"shot_\d+", column)]
    return sorted(columns, key=shot_column_number)


def multiple_match(duration_sec: float) -> dict:
    if pd.isna(duration_sec) or duration_sec <= 0:
        return {"matches": False}

    multiple = max(1, int(round(duration_sec / BASE_LENGTH_SECONDS)))
    expected_sec = multiple * BASE_LENGTH_SECONDS
    error_sec = duration_sec - expected_sec
    abs_error_sec = abs(error_sec)

    return {
        "matches": abs_error_sec <= TOLERANCE_SECONDS,
        "multiple": multiple,
        "expected_sec": expected_sec,
        "error_sec": error_sec,
        "abs_error_sec": abs_error_sec,
    }


def load_inputs() -> tuple[pd.DataFrame, pd.DataFrame, list[str]]:
    if not WIDE_CSV.exists():
        raise FileNotFoundError(f"Missing wide CSV: {WIDE_CSV}")
    if not LONG_CSV.exists():
        raise FileNotFoundError(f"Missing long CSV: {LONG_CSV}")

    wide_df = pd.read_csv(WIDE_CSV)
    long_df = pd.read_csv(LONG_CSV)
    shot_columns = sorted_shot_columns(wide_df)

    required_long_columns = {"film", "shot_index", "duration_sec", "start_sec", "end_sec", "video_path"}
    missing = required_long_columns - set(long_df.columns)
    if missing:
        raise ValueError(f"Long CSV is missing required columns: {sorted(missing)}")

    if "film" not in wide_df.columns:
        raise ValueError("Wide CSV is missing required column: film")
    if not shot_columns:
        raise ValueError("Wide CSV has no shot_#### columns")

    return wide_df, long_df, shot_columns


wide_df, long_df, shot_columns = load_inputs()
print(f"Loaded {len(wide_df)} films from wide CSV.")
print(f"Loaded {len(long_df)} shot rows from long CSV.")
print(f"Output suffix: {output_suffix()}")

Loaded 454 films from wide CSV.
Loaded 193593 shot rows from long CSV.
Output suffix: shot3_len3_tlr0p2


In [3]:
def film_shots_from_wide(row: pd.Series, shot_columns: list[str]) -> pd.DataFrame:
    rows = []
    for column in shot_columns:
        duration = row[column]
        if pd.isna(duration):
            continue
        shot_index = shot_column_number(column)
        match = multiple_match(float(duration))
        rows.append({
            "film": row["film"],
            "shot_index": shot_index,
            "duration_sec": float(duration),
            **match,
        })
    return pd.DataFrame(rows)


def quote_long_segment(film: str, start_shot: int, end_shot: int, long_df: pd.DataFrame) -> pd.DataFrame:
    segment = long_df[
        (long_df["film"] == film)
        & (long_df["shot_index"] >= start_shot)
        & (long_df["shot_index"] <= end_shot)
    ].copy()
    return segment.sort_values("shot_index")


def make_candidate_row(film: str, segment: pd.DataFrame, long_df: pd.DataFrame) -> dict:
    start_shot = int(segment["shot_index"].min())
    end_shot = int(segment["shot_index"].max())
    quoted = quote_long_segment(film, start_shot, end_shot, long_df)

    durations = [round(float(value), 3) for value in segment["duration_sec"].tolist()]
    multiples = [int(value) for value in segment["multiple"].tolist()]
    expected = [round(float(value), 3) for value in segment["expected_sec"].tolist()]
    errors = [round(float(value), 3) for value in segment["error_sec"].tolist()]
    abs_errors = [float(value) for value in segment["abs_error_sec"].tolist()]

    if quoted.empty:
        start_sec = math.nan
        end_sec = math.nan
        video_path = ""
    else:
        start_sec = float(quoted["start_sec"].min())
        end_sec = float(quoted["end_sec"].max())
        video_path = str(quoted["video_path"].iloc[0])

    return {
        "film": film,
        "start_shot": start_shot,
        "end_shot": end_shot,
        "shot_count": int(len(segment)),
        "start_sec": round(start_sec, 3) if not math.isnan(start_sec) else math.nan,
        "end_sec": round(end_sec, 3) if not math.isnan(end_sec) else math.nan,
        "duration_total_sec": round(end_sec - start_sec, 3) if not math.isnan(start_sec) and not math.isnan(end_sec) else math.nan,
        "base_length_sec": BASE_LENGTH_SECONDS,
        "tolerance_sec": TOLERANCE_SECONDS,
        "durations_sec": ";".join(str(value) for value in durations),
        "multiples": ";".join(str(value) for value in multiples),
        "expected_sec": ";".join(str(value) for value in expected),
        "errors_sec": ";".join(str(value) for value in errors),
        "max_abs_error_sec": round(max(abs_errors), 3),
        "mean_abs_error_sec": round(sum(abs_errors) / len(abs_errors), 3),
        "video_path": video_path,
    }


def find_maximal_runs(film_shots: pd.DataFrame, long_df: pd.DataFrame) -> list[dict]:
    candidates = []
    if film_shots.empty:
        return candidates

    current_rows = []
    previous_shot_index = None

    for _, shot in film_shots.sort_values("shot_index").iterrows():
        contiguous = previous_shot_index is None or int(shot["shot_index"]) == previous_shot_index + 1
        if bool(shot["matches"]) and contiguous:
            current_rows.append(shot)
        else:
            if len(current_rows) >= MIN_CONSECUTIVE_SHOTS:
                segment = pd.DataFrame(current_rows)
                candidates.append(make_candidate_row(str(segment["film"].iloc[0]), segment, long_df))
            current_rows = [shot] if bool(shot["matches"]) else []
        previous_shot_index = int(shot["shot_index"])

    if len(current_rows) >= MIN_CONSECUTIVE_SHOTS:
        segment = pd.DataFrame(current_rows)
        candidates.append(make_candidate_row(str(segment["film"].iloc[0]), segment, long_df))

    return candidates


def find_all_windows(film_shots: pd.DataFrame, long_df: pd.DataFrame) -> list[dict]:
    candidates = []
    film_shots = film_shots.sort_values("shot_index").reset_index(drop=True)
    for start in range(0, len(film_shots) - MIN_CONSECUTIVE_SHOTS + 1):
        window = film_shots.iloc[start:start + MIN_CONSECUTIVE_SHOTS].copy()
        expected_indices = list(range(int(window["shot_index"].iloc[0]), int(window["shot_index"].iloc[0]) + MIN_CONSECUTIVE_SHOTS))
        actual_indices = [int(value) for value in window["shot_index"].tolist()]
        if actual_indices != expected_indices:
            continue
        if bool(window["matches"].all()):
            candidates.append(make_candidate_row(str(window["film"].iloc[0]), window, long_df))
    return candidates


def find_candidates_for_film(row: pd.Series, shot_columns: list[str], long_df: pd.DataFrame) -> pd.DataFrame:
    film_shots = film_shots_from_wide(row, shot_columns)
    if RECORD_MODE == "maximal_runs":
        candidates = find_maximal_runs(film_shots, long_df)
    elif RECORD_MODE == "all_windows":
        candidates = find_all_windows(film_shots, long_df)
    else:
        raise ValueError("RECORD_MODE must be 'maximal_runs' or 'all_windows'")
    return pd.DataFrame(candidates)

In [4]:
all_candidates = []
suffix = output_suffix()

for _, film_row in wide_df.sort_values("film").iterrows():
    film = str(film_row["film"])
    film_candidates = find_candidates_for_film(film_row, shot_columns, long_df)
    if film_candidates.empty:
        film_candidates = pd.DataFrame(columns=[
            "film", "start_shot", "end_shot", "shot_count", "start_sec", "end_sec",
            "duration_total_sec", "base_length_sec", "tolerance_sec", "durations_sec",
            "multiples", "expected_sec", "errors_sec", "max_abs_error_sec",
            "mean_abs_error_sec", "video_path",
        ])
    else:
        all_candidates.append(film_candidates)

    if WRITE_PER_FILM_CSV:
        output_path = OUTPUT_DIR / f"{safe_filename_part(film)}_{suffix}.csv"
        film_candidates.to_csv(output_path, index=False, encoding="utf-8-sig")

if all_candidates:
    combined_df = pd.concat(all_candidates, ignore_index=True)
    combined_df = combined_df.sort_values(
        ["film", "start_shot", "end_shot"], kind="stable"
    ).reset_index(drop=True)
else:
    combined_df = pd.DataFrame(columns=[
        "film", "start_shot", "end_shot", "shot_count", "start_sec", "end_sec",
        "duration_total_sec", "base_length_sec", "tolerance_sec", "durations_sec",
        "multiples", "expected_sec", "errors_sec", "max_abs_error_sec",
        "mean_abs_error_sec", "video_path",
    ])

if WRITE_COMBINED_CSV:
    combined_path = OUTPUT_DIR / f"all_films_{suffix}.csv"
    combined_df.to_csv(combined_path, index=False, encoding="utf-8-sig")
    print(f"Combined output: {combined_path.resolve()}")

print(f"Films scanned: {len(wide_df)}")
print(f"Candidate rows: {len(combined_df)}")
print(f"Per-film files written: {len(wide_df) if WRITE_PER_FILM_CSV else 0}")
display(combined_df.head(20))

Combined output: /Users/caoshuangyuan/Desktop/Workspace/DirtyPilot/metric_montage_results/all_films_shot3_len3_tlr0p2.csv
Films scanned: 454
Candidate rows: 307
Per-film files written: 454


,film,start_shot,end_shot,shot_count,start_sec,end_sec,duration_total_sec,base_length_sec,tolerance_sec,durations_sec,multiples,expected_sec,errors_sec,max_abs_error_sec,mean_abs_error_sec,video_path
0,51号兵站,65,67,3,677.491,758.422,80.931,3.0,0.2,62.888;3.16;14.882,21;1;5,63.0;3.0;15.0,-0.112;0.16;-0.118,0.160,0.130,/Volumes/Extreme SSD/Video/Chinese1949-66/51号兵...
1,51号兵站,433,436,4,4244.050,4307.059,63.009,3.0,0.2,12.082;21.083;2.84;27.004,4;7;1;9,12.0;21.0;3.0;27.0,0.082;0.083;-0.16;0.004,0.160,0.082,/Volumes/Extreme SSD/Video/Chinese1949-66/51号兵...
2,51号兵站,477,480,4,4491.243,4539.010,47.767,3.0,0.2,2.84;14.922;11.962;18.042,1;5;4;6,3.0;15.0;12.0;18.0,-0.16;-0.078;-0.038;0.042,0.160,0.080,/Volumes/Extreme SSD/Video/Chinese1949-66/51号兵...
3,一天一夜,202,204,3,2400.638,2421.601,20.963,3.0,0.2,12.002;5.921;3.04,4;2;1,12.0;6.0;3.0,0.002;-0.079;0.04,0.079,0.040,/Volumes/Extreme SSD/Video/Chinese1949-66/一天一夜...
4,一文钱,58,62,5,75.970,93.973,18.003,3.0,0.2,3.0;3.0;6.001;3.0;3.0,1;1;2;1;1,3.0;3.0;6.0;3.0;3.0,0.0;0.0;0.001;0.0;0.0,0.001,0.000,/Volumes/Extreme SSD/Video/Chinese1949-66/一文钱.mp4
5,一日千里,36,38,3,219.310,279.078,59.768,3.0,0.2,20.923;11.882;26.964,7;4;9,21.0;12.0;27.0,-0.077;-0.118;-0.036,0.118,0.077,/Volumes/Extreme SSD/Video/Chinese1949-66/一日千里...
6,一日千里,307,309,3,3963.779,4029.588,65.809,3.0,0.2,18.002;3.0;44.806,6;1;15,18.0;3.0;45.0,0.002;0.0;-0.194,0.194,0.065,/Volumes/Extreme SSD/Video/Chinese1949-66/一日千里...
7,一百个放心,66,68,3,350.847,362.809,11.962,3.0,0.2,3.0;5.961;3.0,1;2;1,3.0;6.0;3.0,0.0;-0.039;0.0,0.039,0.013,/Volumes/Extreme SSD/Video/Chinese1949-66/一百个放...
8,一百个放心,233,236,4,1405.749,1423.752,18.003,3.0,0.2,3.0;3.0;3.0;9.001,1;1;1;3,3.0;3.0;3.0;9.0,0.0;0.0;0.0;0.001,0.001,0.000,/Volumes/Extreme SSD/Video/Chinese1949-66/一百个放...
9,一贯害人道,542,544,3,6266.464,6284.426,17.962,3.0,0.2,12.122;3.0;2.84,4;1;1,12.0;3.0;3.0,0.122;0.0;-0.16,0.160,0.094,/Volumes/Extreme SSD/Video/Chinese1949-66/一贯害人...


In [5]:
# Optional inspection helper: show the exact long-table shot rows for one candidate.
# Change CANDIDATE_ROW_INDEX after reviewing combined_df above.
CANDIDATE_ROW_INDEX = 0

if len(combined_df) == 0:
    print("No candidates to inspect.")
else:
    candidate = combined_df.iloc[CANDIDATE_ROW_INDEX]
    quoted = quote_long_segment(
        str(candidate["film"]),
        int(candidate["start_shot"]),
        int(candidate["end_shot"]),
        long_df,
    )
    print(
        f"{candidate['film']} shots {int(candidate['start_shot'])}-"
        f"{int(candidate['end_shot'])}, time {candidate['start_sec']}-"
        f"{candidate['end_sec']} sec"
    )
    display(quoted)

51号兵站 shots 65-67, time 677.491-758.422 sec


,film,shot_index,duration_sec,start_sec,end_sec,video_path
64,51号兵站,65,62.888,677.491,740.379,/Volumes/Extreme SSD/Video/Chinese1949-66/51号兵...
65,51号兵站,66,3.160,740.379,743.540,/Volumes/Extreme SSD/Video/Chinese1949-66/51号兵...
66,51号兵站,67,14.882,743.540,758.422,/Volumes/Extreme SSD/Video/Chinese1949-66/51号兵...
